# ⌛ **AI Sandbox**

_Practice makes perfect_

<img src="assets/sandbox.jpg" alt="AI Sandbox Banner" width="95%">


**Sample Workflows:**

- Basic data exploration (i.e., handling outliers, missing values, visualizations)
- Build sample ML project from dataset to evaluation

---
---

In [1]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
df = data.frame
df

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,0
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,0
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,0
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,0


In [2]:
len(df['target'].value_counts())

2

In [3]:
X = df.drop(columns=["target"])
y = df["target"]

X.shape, y.shape

((569, 30), (569,))

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=True, stratify=y, random_state=42)

X_train.shape, X_val.shape, y_train.shape, y_val.shape

((455, 30), (114, 30), (455,), (114,))

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit_transform(X_train)
scaler.transform(X_val)

array([[ 1.56851278,  2.16401585,  1.74286587, ...,  1.20214432,
        -0.14043064,  0.90036171],
       [-0.84027641, -0.5970672 , -0.8741735 , ..., -1.10124064,
        -0.81429139, -0.71323608],
       [-0.07072262,  1.19138742,  0.03202768, ...,  0.57255843,
         1.14997397,  1.86841162],
       ...,
       [ 0.3583966 , -0.44935493,  0.48264098, ...,  1.63386036,
         0.54117564,  1.91191948],
       [-0.41401799,  1.07321761, -0.42314565, ...,  0.06139464,
        -0.1280378 ,  0.64747227],
       [-0.6829327 , -0.69932953, -0.66607058, ..., -0.23615965,
        -0.16211811, -0.20147486]])

In [6]:
import torch

# Make sure features are float32
X_train = torch.from_numpy(X_train.values).float()
X_val   = torch.from_numpy(X_val.values).float()

# Make sure labels are long (for classification)
y_train = torch.from_numpy(y_train.values).long()
y_val   = torch.from_numpy(y_val.values).long()

In [7]:
import torch
import torch.nn as nn

class SimpleNN(nn.Module):

    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size)
        )

    def forward(self, X):
        return self.model(X)
    
model = SimpleNN(input_size=X_train.shape[1], hidden_size=16, output_size=2)
model

SimpleNN(
  (model): Sequential(
    (0): Linear(in_features=30, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=2, bias=True)
  )
)

In [8]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

n_epochs = 100
for epoch in range(n_epochs):

    # Forward
    model.train()
    y_train_pred = model(X_train)
    train_loss = loss_fn(y_train_pred, y_train)
    
    # Backward
    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()

    # Eval
    model.eval()
    with torch.no_grad():
        y_val_pred = model(X_val)
        test_loss = loss_fn(y_val_pred, y_val)

    print(f"Epoch {epoch+1}; Train Loss: {train_loss:.4f}; Test Loss: {test_loss:.4f}")

Epoch 1; Train Loss: 6.9834; Test Loss: 6.1066
Epoch 2; Train Loss: 5.0300; Test Loss: 4.4879
Epoch 3; Train Loss: 3.5618; Test Loss: 3.7819
Epoch 4; Train Loss: 3.0656; Test Loss: 3.9063
Epoch 5; Train Loss: 3.2432; Test Loss: 4.1724
Epoch 6; Train Loss: 3.5312; Test Loss: 4.2707
Epoch 7; Train Loss: 3.6625; Test Loss: 4.1801
Epoch 8; Train Loss: 3.5909; Test Loss: 3.9320
Epoch 9; Train Loss: 3.3491; Test Loss: 3.5746
Epoch 10; Train Loss: 2.9933; Test Loss: 3.1720
Epoch 11; Train Loss: 2.5953; Test Loss: 2.7972
Epoch 12; Train Loss: 2.2319; Test Loss: 2.5497
Epoch 13; Train Loss: 1.9962; Test Loss: 2.5144
Epoch 14; Train Loss: 1.9547; Test Loss: 2.6226
Epoch 15; Train Loss: 2.0345; Test Loss: 2.6540
Epoch 16; Train Loss: 2.0585; Test Loss: 2.4940
Epoch 17; Train Loss: 1.9218; Test Loss: 2.1637
Epoch 18; Train Loss: 1.6406; Test Loss: 1.7767
Epoch 19; Train Loss: 1.3054; Test Loss: 1.4675
Epoch 20; Train Loss: 1.0300; Test Loss: 1.2628
Epoch 21; Train Loss: 0.8688; Test Loss: 1.1271
E

---
---
---